# ROPAgen Document Evaluation
## LLM-as-Judge (Reference-Free Quality)

This notebook is the **third evaluation layer** for the thesis. It uses an LLM as an automated judge to assess the quality of ROPA documents generated by study participants — not by comparing them to a reference text, but by asking the model directly whether each document is legally sound and matches the scenario.

### Where this fits

| Layer | Folder | What it measures |
|---|---|---|
| 1. User survey | `python/survey/` | Subjective usability (SUS), cognitive load (NASA-TLX), confidence, mode preferences |
| 2. NLP metrics | `python/metrics/` | Lexical / semantic **similarity** between each generated ROPA and a single expert-written reference (BLEU, ROUGE, METEOR, BERTScore, SBERT) |
| **3. LLM-as-judge (this notebook)** | `python/judge/` | **Reference-free quality** — each document judged directly against the scenario and Art. 30 GDPR requirements |

### Why a third layer?

NLP metrics answer "how similar is this document to a single expert reference?" That is useful but indirect: a document can be legally sound yet score low on BLEU because it uses different wording, or it can mimic the reference surface while missing required Art. 30 elements. By asking an LLM judge to grade each document against the **task** — the scenario and GDPR requirements — we get a quality signal that is independent of any reference text.

---

### Pipeline overview (6 steps)

**1. Setup**
Load the 113 ROPA documents generated by study participants (via ropagen, powered by Mistral Large 3), the scenario text, and the prompt configuration. Also initialise the OpenRouter API client used for all model calls.

**2. Sample run**
Before scoring everything, judge one document per mode (Form, Ask, Chat) and inspect the raw output. This is a sanity check — it lets you verify the rubric and prompt produce sensible scores before spending API budget on all 113 documents.

**3. Full sweep**
Gemini 3.1 Pro Preview scores every document on five dimensions (completeness, scenario faithfulness, legal correctness, hallucination inverse, overall) using a 1–5 Likert scale. Each result is appended to a disk cache as it is produced, so the sweep can be interrupted and resumed without losing work.

**4. Aggregation & visualisation**
The cache is parsed into two CSV files — one with every score per document per metric, one with per-mode summary statistics (mean, median, std). A boxplot figure is saved showing score distributions across the three modes.

**5. Judge reflection**
Gemini 3.1 Pro Preview is asked to step back and interpret its own scoring results. It receives the per-mode summary statistics, survey confidence data (how much users trusted each mode), and six concrete document examples (best and worst per mode). It writes a 500–700 word qualitative narrative addressing three research questions: how LLM support level affects quality, whether all Art. 30 fields are covered, and whether there is a gap between user confidence and actual document quality.

**6. Meta-judge reflection**
Finally, Claude Sonnet 4.6 receives all five inputs — judge scores, NLP metrics, survey confidence, the judge's own reflection, and the concrete examples — and writes a methodological triangulation: where the two evaluation layers agree, where they diverge, and what that means for all four research questions including the BERTScore-based quality comparison.

## Inputs & outputs

### Inputs

| File | Description |
|---|---|
| `python/metrics/docs/documents.csv` | 113 ROPA documents (id, user_id, ai_mode, document text) |
| `python/metrics/output/document_results.csv` | Per-document NLP metric scores from the metrics notebook |
| `python/survey/docs/data.csv` | Raw survey responses (SUS, NASA-TLX, AI quality questions, confidence) |
| `python/judge/scenario.txt` | The standardised GDPR scenario shown to all participants |
| `python/judge/prompt.json` | Prompt templates and model config for judge, judge_reflection, and meta_judge |

### Outputs

```
python/judge/output/
├── judge_cache.jsonl           # raw Gemini 3.1 Pro responses (113 lines)
├── judge_sample.json           # 3-doc sample used to validate the rubric
├── judge_results.csv           # per-doc × per-metric scores + rationales
├── judge_summary.csv           # per-mode summary stats
├── judge_metrics_boxplot.png   # figure for the thesis
├── judge_reflection.md         # Gemini's qualitative RQ narrative + Fallbeispiele
└── meta_judge_reflection.md    # Claude Sonnet 4.6's written analysis
```

In [1]:
# ── Setup: paths, env, data, prompt config ────────────────────────────────────
import os
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Resolve paths relative to this notebook, not the CWD, so it runs from anywhere
JUDGE_DIR = Path.cwd()
if JUDGE_DIR.name != "judge":
    # Fallback when notebook is executed from the repo root
    JUDGE_DIR = Path("python/judge").resolve()

OUT_DIR = JUDGE_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# .env lives next to the notebook and holds OPENROUTER_API_KEY
load_dotenv(JUDGE_DIR / ".env")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY is not set in python/judge/.env"

# Inputs
DOCUMENTS_CSV = (JUDGE_DIR.parent / "metrics" / "docs" / "documents.csv").resolve()
NLP_RESULTS_CSV = (JUDGE_DIR.parent / "metrics" / "output" / "document_results.csv").resolve()
SCENARIO_TXT  = (JUDGE_DIR / "scenario.txt").resolve()
PROMPT_JSON   = (JUDGE_DIR / "prompt.json").resolve()

# Outputs
CACHE_JSONL   = OUT_DIR / "judge_cache.jsonl"
RESULTS_CSV   = OUT_DIR / "judge_results.csv"
SUMMARY_CSV   = OUT_DIR / "judge_summary.csv"
BOXPLOT_PNG   = OUT_DIR / "judge_metrics_boxplot.png"
META_MD       = OUT_DIR / "meta_judge_reflection.md"
SAMPLE_JSON   = OUT_DIR / "judge_sample.json"

# Load inputs
documents_df = pd.read_csv(DOCUMENTS_CSV)
scenario_text = SCENARIO_TXT.read_text(encoding="utf-8")
prompt_cfg = json.loads(PROMPT_JSON.read_text(encoding="utf-8"))

print(f"documents.csv   : {documents_df.shape[0]} rows, cols = {list(documents_df.columns)}")
print(f"ai_mode counts  : {dict(documents_df['ai_mode'].value_counts())}")
print(f"scenario.txt    : {len(scenario_text)} chars")
print(f"judge model     : {prompt_cfg['judge']['model']}")
print(f"meta-judge model: {prompt_cfg['meta_judge']['model']}")


documents.csv   : 113 rows, cols = ['id', 'user_id', 'ai_mode', 'document']
ai_mode counts  : {'chat': np.int64(39), 'form': np.int64(37), 'ask': np.int64(37)}
scenario.txt    : 1673 chars
judge model     : google/gemini-3.1-pro-preview
meta-judge model: anthropic/claude-sonnet-4.6


In [2]:
# ── Judge client + per-document call with retries and disk cache ─────────────
import json
from openai import OpenAI
from openai import APIError, APIConnectionError, RateLimitError, APITimeoutError

JUDGE_METRICS = prompt_cfg["judge"]["metrics"]  # canonical ordering for results

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)


def _load_cache_ids(cache_path: Path) -> set:
    """Read the append-only JSONL cache and return set of already-judged ids."""
    if not cache_path.exists():
        return set()
    done = set()
    with cache_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                done.add(int(obj["id"]))
            except Exception:
                # Skip malformed lines silently - the cache is best-effort
                continue
    return done


def _append_cache(cache_path: Path, record: dict) -> None:
    with cache_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def _build_messages(document_text: str) -> list:
    judge_cfg = prompt_cfg["judge"]
    user_msg = judge_cfg["user_template"].format(
        scenario=scenario_text,
        document=document_text,
    )
    return [
        {"role": "system", "content": judge_cfg["system"]},
        {"role": "user",   "content": user_msg},
    ]


def judge_document(document_text: str,
                   max_retries: int = 4,
                   base_backoff: float = 2.0) -> dict:
    """Call the judge once. Returns parsed JSON dict with 5 metric fields.

    Retries on transient errors (connection, rate-limit, timeout, 5xx)
    with exponential backoff. Raises the last exception if all retries fail.
    """
    judge_cfg = prompt_cfg["judge"]
    messages = _build_messages(document_text)

    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=judge_cfg["model"],
                temperature=judge_cfg.get("temperature", 0.0),
                messages=messages,
                response_format=judge_cfg["response_schema"],
            )
            content = resp.choices[0].message.content
            # The judge is instructed to return JSON only, but defensively
            # strip any stray Markdown fences before parsing.
            content = content.strip()
            if content.startswith("```"):
                content = content.strip("`")
                # Drop optional language tag like "json\n"
                nl = content.find("\n")
                if nl != -1:
                    content = content[nl + 1 :]
                content = content.rstrip("`").strip()
            return json.loads(content)
        except (APIConnectionError, APITimeoutError, RateLimitError) as e:
            last_err = e
        except APIError as e:
            # Only retry on 5xx; re-raise on 4xx (likely permanent)
            status = getattr(e, "status_code", None) or getattr(e, "status", None)
            if status is not None and 400 <= int(status) < 500:
                raise
            last_err = e
        except json.JSONDecodeError as e:
            # Model occasionally returns malformed JSON - retry once or twice
            last_err = e

        sleep_s = base_backoff * (2 ** attempt)
        time.sleep(sleep_s)

    raise RuntimeError(f"judge_document failed after {max_retries} attempts: {last_err}")


print("client ready:", client.base_url)
print("cache file  :", CACHE_JSONL)
print("already done:", len(_load_cache_ids(CACHE_JSONL)), "ids")


client ready: https://openrouter.ai/api/v1/
cache file  : C:\Dev\ropagen-bert\python\judge\output\judge_cache.jsonl
already done: 0 ids


In [3]:
# ── Sample judge run: first document per mode ────────────────────────────────
sample_rows = (
    documents_df.sort_values(["ai_mode", "id"])
                .groupby("ai_mode", as_index=False)
                .first()
)
sample_rows = sample_rows[["id", "user_id", "ai_mode", "document"]]
print("Sample documents selected:")
print(sample_rows[["id", "user_id", "ai_mode"]].to_string(index=False))

samples = []
for _, row in sample_rows.iterrows():
    parsed = judge_document(row["document"])
    rec = {
        "id": int(row["id"]),
        "user_id": str(row["user_id"]),
        "ai_mode": str(row["ai_mode"]),
        "judge": parsed,
    }
    samples.append(rec)
    print(f"\n--- id={rec['id']} mode={rec['ai_mode']} user={rec['user_id']} ---")
    print(json.dumps(parsed, indent=2, ensure_ascii=False))

# Persist for inspection without another API call
SAMPLE_JSON.write_text(json.dumps(samples, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nSaved sample -> {SAMPLE_JSON}")



--- id=3 mode=form user=EHN0215 ---
{
  "completeness": {
    "score": 5,
    "rationale": "The document successfully covers all mandatory elements required by Article 30(1) GDPR, including controller details, purposes, data categories, recipients, retention periods, and TOMs."
  },
  "scenario_faithfulness": {
    "score": 2,
    "rationale": "Directly contradicts the scenario by including 'sick days' (which were explicitly excluded) and stating 3- to 10-year retention periods instead of the specified 3 months."
  },
  "legal_correctness": {
    "score": 3,
    "rationale": "Uses correct GDPR terminology and standard legal bases for employment, but unnecessarily includes irrelevant laws like the Infection Protection Act (IfSG) which contradicts the exclusion of medical data."
  },
  "hallucination_inverse": {
    "score": 2,
    "rationale": "Contains significant hallucinations, including invented retention periods, unmentioned software systems (e.g., HR-Software, Enterprise Systems)

In [4]:
# ── Full judge sweep with disk caching ───────────────────────────────────────
from tqdm.auto import tqdm

done_ids = _load_cache_ids(CACHE_JSONL)
pending  = documents_df[~documents_df["id"].astype(int).isin(done_ids)].copy()
print(f"cache has {len(done_ids)} ids; {len(pending)} documents still to judge")

failures = []
for _, row in tqdm(pending.iterrows(), total=len(pending), desc="Judging"):
    doc_id = int(row["id"])
    try:
        parsed = judge_document(row["document"])
        record = {
            "id": doc_id,
            "user_id": str(row["user_id"]),
            "ai_mode": str(row["ai_mode"]),
            "judge": parsed,
        }
        _append_cache(CACHE_JSONL, record)
    except Exception as e:
        failures.append({"id": doc_id, "error": repr(e)})

print(f"\nDone. Total cached ids: {len(_load_cache_ids(CACHE_JSONL))}")
if failures:
    print(f"{len(failures)} failures (re-run the cell to retry):")
    for f in failures[:10]:
        print(" ", f)



Done. Total cached ids: 113


In [5]:
# ── Flatten cache to long-format CSV + per-mode summary ──────────────────────
import pandas as pd

long_rows = []
with CACHE_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        for metric in JUDGE_METRICS:
            entry = rec["judge"].get(metric, {})
            long_rows.append({
                "id":        int(rec["id"]),
                "user_id":   rec["user_id"],
                "ai_mode":   rec["ai_mode"],
                "metric":    metric,
                "score":     int(entry.get("score")) if entry.get("score") is not None else None,
                "rationale": entry.get("rationale", ""),
            })

judge_results = pd.DataFrame(long_rows)
judge_results = judge_results.sort_values(["id", "metric"]).reset_index(drop=True)
judge_results.to_csv(RESULTS_CSV, index=False)
print(f"Saved {RESULTS_CSV}  ({len(judge_results)} rows)")

# Per-mode summary: mean / median / std per metric
summary = (
    judge_results
    .groupby(["ai_mode", "metric"])["score"]
    .agg(["count", "mean", "median", "std"])
    .round(3)
    .reset_index()
    .rename(columns={"count": "n"})
)
summary.to_csv(SUMMARY_CSV, index=False)
print(f"Saved {SUMMARY_CSV}")

# Pivot for readability
pivot = summary.pivot(index="metric", columns="ai_mode", values="mean")
pivot = pivot.reindex(JUDGE_METRICS)
print("\nMean score per mode (higher is better, scale 1-5):")
print(pivot.to_string())


Saved C:\Dev\ropagen-bert\python\judge\output\judge_results.csv  (565 rows)
Saved C:\Dev\ropagen-bert\python\judge\output\judge_summary.csv

Mean score per mode (higher is better, scale 1-5):
ai_mode                  ask   chat   form
metric                                    
completeness           3.838  3.179  4.838
scenario_faithfulness  2.297  2.487  1.838
legal_correctness      3.432  3.154  3.595
hallucination_inverse  2.405  2.538  1.649
overall                2.486  2.615  2.405


In [6]:
# ── Per-metric judge boxplots (style matched to metrics notebook) ────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

JUDGE_METRIC_META = [
    ("completeness",          "Completeness  (Art. 30 coverage)"),
    ("scenario_faithfulness", "Scenario Faithfulness"),
    ("legal_correctness",     "Legal Correctness"),
    ("hallucination_inverse", "Hallucination  (inverse - higher = fewer)"),
    ("overall",               "Overall Quality"),
]

GROUPS      = ["All",     "form",    "ask",     "chat"]
BOX_COLORS  = ["#FFD59A", "#A7C7E7", "#FFF9AE", "#E0BBE4"]
MEAN_COLORS = ["#FF8C00", "#0000FF", "#FFD700", "#8A2BE2"]


def _scores(metric: str, mode_key: str) -> np.ndarray:
    df_m = judge_results[judge_results["metric"] == metric]
    if mode_key != "All":
        df_m = df_m[df_m["ai_mode"] == mode_key]
    return df_m["score"].dropna().values.astype(float)


def _draw_metric(ax, metric, title, add_legend=False):
    group_data = [_scores(metric, g) for g in GROUPS]

    bp = ax.boxplot(
        group_data,
        positions=range(len(GROUPS)),
        widths=0.52,
        patch_artist=True,
        showfliers=True,
        medianprops=dict(color="black", linewidth=0.9),
        whiskerprops=dict(color="black", linewidth=0.8),
        capprops=dict(color="black", linewidth=0.8),
        boxprops=dict(linewidth=0.8),
        flierprops=dict(marker="o", markerfacecolor="grey",
                        markeredgecolor="grey", markersize=4, alpha=0.55),
    )

    for patch, box_col in zip(bp["boxes"], BOX_COLORS):
        patch.set_facecolor(box_col)
        patch.set_edgecolor("black")

    for i, (data, mean_col) in enumerate(zip(group_data, MEAN_COLORS)):
        if len(data) == 0:
            continue
        ax.scatter(i, np.mean(data), marker="D", s=90,
                   color=mean_col, edgecolors="black", linewidths=0.8, zorder=6)

    ax.set_xticks(range(len(GROUPS)))
    ax.set_xticklabels(GROUPS, fontsize=12)

    # 1-5 Likert: fixed y range with a little headroom
    ax.set_ylim(0.6, 5.4)
    ax.set_yticks([1, 2, 3, 4, 5])

    ax.set_ylabel("Score (1-5)", fontsize=11)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=12)
    ax.yaxis.grid(True, linestyle="--", linewidth=0.8, alpha=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

    box_patches = [
        mpatches.Patch(facecolor=bc, edgecolor="black", label=g)
        for g, bc in zip(GROUPS, BOX_COLORS)
    ]
    mean_handles = [
        plt.Line2D([0], [0], marker="D", color="w",
                   markerfacecolor=mc, markeredgecolor="black",
                   markersize=8, label=f"{g} mean")
        for g, mc in zip(GROUPS, MEAN_COLORS)
    ]
    if add_legend:
        ax.legend(handles=box_patches + mean_handles, loc="lower right",
                  fontsize=9, ncol=2, framealpha=0.88, edgecolor="lightgrey")


# 2 x 3 grid, last slot unused (mirrors the NLP-metrics figure layout)
fig, axes = plt.subplots(2, 3, figsize=(22, 12))
axes = axes.flatten()

for i, (metric, title) in enumerate(JUDGE_METRIC_META):
    _draw_metric(axes[i], metric, title, add_legend=(i == 0))

fig.delaxes(axes[5])

plt.suptitle(
    "LLM-as-Judge Metrics  -  Distribution by AI Generation Mode  (Per Document)",
    fontsize=16, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.savefig(BOXPLOT_PNG, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {BOXPLOT_PNG}")


Saved -> C:\Dev\ropagen-bert\python\judge\output\judge_metrics_boxplot.png


In [8]:
# ── Judge Reflection: qualitative narrative by Gemini on its own scoring results ──
import numpy as np
import pandas as pd

SURVEY_CSV    = (JUDGE_DIR.parent / "survey" / "docs" / "data.csv").resolve()
REFLECTION_MD = OUT_DIR / "judge_reflection.md"

# ── 1. Judge summary (reuse summary var from cc4d82e3, or reload from CSV) ───
if "summary" in dir():
    _pm = summary.pivot(index="metric", columns="ai_mode", values="mean").reindex(JUDGE_METRICS)
else:
    _s  = pd.read_csv(SUMMARY_CSV)
    _pm = _s.pivot(index="metric", columns="ai_mode", values="mean").reindex(JUDGE_METRICS)
judge_summary_str = "Mean per mode:\n" + _pm.round(3).to_string() + "\n"

# ── 2. Survey confidence data ──────────────────────────────────────────────
AI_Q_LABELS = {
    "1": "Helpfulness", "2": "Speed", "3": "Transparency",
    "4": "Trust",       "5": "Legal confidence", "6": "Reuse intent",
}
survey_raw = pd.read_csv(SURVEY_CSV, sep=";")
survey_raw = survey_raw[survey_raw["p_0001"] >= 1]

conf_records = []
for _, row in survey_raw.iterrows():
    for pos in ["1", "2", "3"]:
        mode = row[f"{pos}_order"]
        if mode not in ["Form", "Ask", "Chat"]:
            continue
        for q_num in AI_Q_LABELS:
            val = row[f"{pos}_{q_num}_ai"]
            if val == -99:
                val = np.nan
            conf_records.append({"mode": mode, "question": AI_Q_LABELS[q_num], "score": val})

conf_df    = pd.DataFrame(conf_records)
conf_means = (
    conf_df.groupby(["mode", "question"])["score"]
    .mean().round(3).unstack("question")
    .reindex(columns=list(AI_Q_LABELS.values()))
)
conf_means["overall_mean"] = conf_means.mean(axis=1).round(3)
survey_confidence_str  = "Per-mode means for 6 AI quality questions (1–5 Likert):\n"
survey_confidence_str += conf_means.to_string() + "\n"
print(survey_confidence_str)

# ── 3. Fallbeispiele selection (best + worst per mode on overall) ───────────
results_df   = pd.read_csv(RESULTS_CSV)
docs_df      = pd.read_csv(DOCUMENTS_CSV)

wide = results_df.pivot_table(
    index=["id", "ai_mode"], columns="metric", values="score"
).reset_index()
wide["total"] = wide[list(JUDGE_METRICS)].sum(axis=1)


def _pick(mode: str, best: bool) -> dict:
    m   = wide[wide["ai_mode"] == mode].sort_values(
        ["overall", "total"], ascending=[not best, not best]
    )
    row    = m.iloc[0]
    doc_id = int(row["id"])
    excerpt = (
        docs_df[docs_df["id"] == doc_id]["document"].iloc[0]
        .strip().lstrip("`").strip()[:400]
    )
    scores = {mn: int(row[mn]) for mn in JUDGE_METRICS}
    rats   = {
        mn: results_df[
            (results_df["id"] == doc_id) & (results_df["metric"] == mn)
        ]["rationale"].iloc[0]
        for mn in ["overall", "hallucination_inverse"]
    }
    return {"id": doc_id, "mode": mode, "label": "best" if best else "worst",
            "scores": scores, "rats": rats, "excerpt": excerpt}


examples = [_pick(m, b) for m in ["form", "ask", "chat"] for b in [False, True]]


def _fmt(exs: list) -> str:
    parts = []
    for ex in exs:
        parts.append(
            f"### {ex['mode'].upper()} — {ex['label'].capitalize()} (id={ex['id']})\n"
            f"Scores: {' | '.join(f'{k}={v}' for k,v in ex['scores'].items())}\n"
            f"Excerpt: {ex['excerpt']}\n"
            f"Rationale (overall): {ex['rats']['overall']}\n"
            f"Rationale (hallucination_inverse): {ex['rats']['hallucination_inverse']}"
        )
    return "\n\n---\n\n".join(parts)


fallbeispiele_str = _fmt(examples)

# ── 4. Call judge reflection model ─────────────────────────────────────────
ref_cfg  = prompt_cfg["judge_reflection"]
ref_resp = client.chat.completions.create(
    model=ref_cfg["model"],
    temperature=ref_cfg.get("temperature", 0.2),
    messages=[
        {"role": "system", "content": ref_cfg["system"]},
        {"role": "user",   "content": ref_cfg["user_template"].format(
            judge_summary=judge_summary_str,
            survey_confidence=survey_confidence_str,
            fallbeispiele=fallbeispiele_str,
        )},
    ],
)
judge_reflection_text = ref_resp.choices[0].message.content
REFLECTION_MD.write_text(judge_reflection_text, encoding="utf-8")
print(f"Saved -> {REFLECTION_MD}\n")
print(judge_reflection_text)

Saved -> C:\Dev\ropagen-bert\python\judge\output\judge_reflection.md

## RQ1: Influence of LLM Support Level on Document Quality

The data reveals a clear trade-off between the level of LLM scaffolding and the resulting factual quality of the generated documents. Counterintuitively, higher scaffolding does not reliably improve overall document quality; rather, it shifts the nature of the AI's performance. The Form mode, which provides the highest level of scaffolding, achieved the highest mean scores for legal correctness (3.595) and completeness (4.838). However, this rigid structure came at a severe cost to factual accuracy. The Form mode recorded the lowest scenario faithfulness (1.838) and the worst performance regarding hallucinations (hallucination_inverse = 1.649, where 1 indicates many hallucinations). 

Conversely, the Chat mode, featuring the lowest scaffolding, yielded the highest overall quality score (2.615) and the best scenario faithfulness (2.487), alongside the fewest 

In [10]:
# ── Meta-judge: build summaries, call Sonnet 4.6, save reflection ────────────
import pandas as pd

# --- Judge summary text (per mode, per metric: mean +/- std, median) ---
judge_pivot_mean   = summary.pivot(index="metric", columns="ai_mode", values="mean").reindex(JUDGE_METRICS)
judge_pivot_median = summary.pivot(index="metric", columns="ai_mode", values="median").reindex(JUDGE_METRICS)
judge_pivot_std    = summary.pivot(index="metric", columns="ai_mode", values="std").reindex(JUDGE_METRICS)

judge_summary_str = "Judge metrics (1-5 Likert, higher = better).\n\n"
judge_summary_str += "Mean per mode:\n"   + judge_pivot_mean.round(3).to_string()   + "\n\n"
judge_summary_str += "Median per mode:\n" + judge_pivot_median.round(3).to_string() + "\n\n"
judge_summary_str += "Std per mode:\n"    + judge_pivot_std.round(3).to_string()    + "\n"

# --- NLP summary text from the metrics notebook's per-document CSV ---
nlp_df = pd.read_csv(NLP_RESULTS_CSV)
NLP_COLS = ["BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR",
            "BERTScore_Precision", "BERTScore_Recall", "BERTScore_F1", "SBERT_ModernBERT"]
nlp_mean   = nlp_df.groupby("ai_mode")[NLP_COLS].mean().T
nlp_median = nlp_df.groupby("ai_mode")[NLP_COLS].median().T
nlp_std    = nlp_df.groupby("ai_mode")[NLP_COLS].std().T

nlp_summary_str = "NLP metrics (0-1, higher = better; BERT/SBERT typically 0.85-1.0).\n\n"
nlp_summary_str += "Mean per mode:\n"   + nlp_mean.round(4).to_string()   + "\n\n"
nlp_summary_str += "Median per mode:\n" + nlp_median.round(4).to_string() + "\n\n"
nlp_summary_str += "Std per mode:\n"    + nlp_std.round(4).to_string()    + "\n"

print("Judge summary preview:")
print(judge_summary_str[:400], "...\n")
print("NLP summary preview:")
print(nlp_summary_str[:400], "...\n")

# --- Meta-judge call ---
meta_cfg = prompt_cfg["meta_judge"]
# ── Fallback: rebuild if judge_reflection cell was skipped ─────────────────────
if "survey_confidence_str" not in dir():
    import numpy as np
    _srv = pd.read_csv(
        (JUDGE_DIR.parent / "survey" / "docs" / "data.csv").resolve(), sep=";"
    )
    _srv = _srv[_srv["p_0001"] >= 1]
    _AI_Q = {"1":"Helpfulness","2":"Speed","3":"Transparency",
              "4":"Trust","5":"Legal confidence","6":"Reuse intent"}
    _recs = []
    for _, _r in _srv.iterrows():
        for _p in ["1","2","3"]:
            _m = _r[f"{_p}_order"]
            if _m not in ["Form","Ask","Chat"]: continue
            for _q,_lbl in _AI_Q.items():
                _v = _r[f"{_p}_{_q}_ai"]
                _recs.append({"mode":_m,"question":_lbl,"score": np.nan if _v==-99 else _v})
    _cm = pd.DataFrame(_recs).groupby(["mode","question"])["score"].mean().round(3).unstack("question")
    _cm["overall_mean"] = _cm.mean(axis=1).round(3)
    survey_confidence_str = "Per-mode means for 6 AI quality questions (1–5 Likert):\n" + _cm.to_string() + "\n"

if "judge_reflection_text" not in dir():
    judge_reflection_text = REFLECTION_MD.read_text(encoding="utf-8") if REFLECTION_MD.exists() else "[run judge_reflection cell first]"

if "fallbeispiele_str" not in dir():
    fallbeispiele_str = "[run judge_reflection cell first]"

# ── Update meta_user format call ─────────────────────────────────────────
meta_user = meta_cfg["user_template"].format(
    judge_summary=judge_summary_str,
    nlp_summary=nlp_summary_str,
    survey_confidence=survey_confidence_str,
    judge_reflection=judge_reflection_text,
    fallbeispiele=fallbeispiele_str,
)
meta_resp = client.chat.completions.create(
    model=meta_cfg["model"],
    temperature=meta_cfg.get("temperature", 0.2),
    messages=[
        {"role": "system", "content": meta_cfg["system"]},
        {"role": "user",   "content": meta_user},
    ],
)
meta_text = meta_resp.choices[0].message.content
META_MD.write_text(meta_text, encoding="utf-8")
print(f"Saved -> {META_MD}")
print("\n--- Meta-judge reflection ---\n")
print(meta_text)


Saved -> C:\Dev\ropagen-bert\python\judge\output\meta_judge_reflection.md

--- Meta-judge reflection ---

# Triangulation Reflection: NLP Metrics and LLM-as-Judge Across Interaction Modes

## 1. Agreement Across Layers

The two evaluation layers converge on one consistent finding: **Chat mode produces documents with the highest surface-level lexical and semantic similarity to the expert reference**, while simultaneously receiving the highest judge scores for scenario faithfulness and hallucination control. Chat leads on ROUGE-1 (0.5106), ROUGE-2 (0.2913), ROUGE-L (0.3892), METEOR (0.3902), BERTScore F1 (0.9041), and SBERT cosine similarity (0.9895), and the judge assigns it the best hallucination_inverse mean (2.538) and highest overall score (2.615). This alignment is substantively coherent: documents that stay closer to scenario-grounded content naturally resemble the expert reference more closely, because the expert also wrote against the same scenario.

A secondary point of agreeme